# Spam Email Detection: Introducing Linear Regression & Text Classification

### Key Objectives:
1. **Explore the SMS Spam Collection dataset**: Binary classification of natural language messages into **Ham** (0) and **Spam** (1).
2. **Understand Text Vectorization (TF-IDF)**: Transforming unstructured text into sparse numerical feature vectors without data leakage.
3. **Contrast 3 Models**:
   - **Linear Regression**: Why Ordinary Least Squares (OLS) conceptually fails for classification (bounds violations, sensitivity to outliers).
   - **Logistic Regression**: The correct probabilistic linear classifier using the Sigmoid link function and Log-Loss.
   - **Multinomial Naive Bayes**: The standard text classification baseline leveraging Bayes' theorem and Laplace smoothing.
4. **Evaluate Asymmetric Costs**: Why **Precision** is the paramount metric in spam filtering.


In [ ]:
import os
import re
import urllib.request
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

RANDOM_STATE = 42
DATA_FILE_PATH = 'sms_spam_data.csv'
PRIMARY_URL = 'https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv'


## 1. Dataset Loading & Exploration

We load the standard **SMS Spam Collection** dataset. If not found locally, we download it and cache it as `sms_spam_data.csv`.


In [ ]:
def load_dataset():
    if os.path.exists(DATA_FILE_PATH):
        print(f"[+] Loading local dataset from '{DATA_FILE_PATH}'...")
        return pd.read_csv(DATA_FILE_PATH)
    
    print("[*] Downloading dataset from primary URL...")
    req = urllib.request.Request(PRIMARY_URL, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=10) as resp:
        content = resp.read().decode('utf-8', errors='ignore')
    
    from io import StringIO
    raw_df = pd.read_csv(StringIO(content), sep='\t', header=None, names=['label', 'text'])
    df = pd.DataFrame({
        'label': raw_df['label'].map(lambda x: 1 if str(x).lower().strip() == 'spam' else 0),
        'text': raw_df['text'].astype(str)
    })
    df.to_csv(DATA_FILE_PATH, index=False)
    print(f"[+] Downloaded and cached {len(df)} records.")
    return df

df = load_dataset()
print("Dataset Shape:", df.shape)
print("\nClass Distribution:")
print(df['label'].value_counts(normalize=True).rename({0: 'Ham (0)', 1: 'Spam (1)'}))
df.head()


## 2. Text Preprocessing & Cleaning

Before vectorization, we apply standard text normalization:
1. **Lowercasing**: Unifies case sensitivity (`"WINNER"` and `"winner"` refer to the same semantic token).
2. **Regex Non-Alphabet Filtering**: Replaces punctuation, numbers, and symbols with whitespace (`re.sub(r'[^a-z\s]', ' ', text)`).
3. **Whitespace Normalization**: Removes duplicate spaces and strips boundary whitespace.


In [ ]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned_text'] = df['text'].apply(clean_text)
df[['text', 'cleaned_text', 'label']].head()


## 3. Stratified Train/Test Split & TF-IDF Vectorization

### Mathematical Foundation of TF-IDF:
$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)$$

1. **Term Frequency (TF)**:
   $$\text{TF}(t, d) = \frac{\text{count}(t, d)}{\sum_{t' \in d} \text{count}(t', d)}$$
2. **Inverse Document Frequency (IDF)**:
   $$\text{IDF}(t, D) = \ln\left(\frac{1 + |D|}{1 + \text{DF}(t)}\right) + 1$$
   *Penalizes ubiquitously frequent words while boosting rare discriminative terms like "prize", "winner", "urgent".*

> **Critical ML Best Practice**: We split into train/test **before** fitting the vectorizer. We `fit_transform` on `X_train` only, and `transform` on `X_test` to prevent **Data Leakage**.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_text'],
    df['label'],
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=df['label']
)

print(f"Train samples: {len(X_train)} (Spam: {y_train.sum()})")
print(f"Test samples:  {len(X_test)} (Spam: {y_test.sum()})")

vectorizer = TfidfVectorizer(max_features=1500, stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"Vocabulary size: {X_train_tfidf.shape[1]} features extracted.")


## 4. Model 1: Linear Regression (OLS on Binary Targets)

### Conceptual Demonstration & Flaws:
Linear Regression fits an unconstrained hyperplane:
$$\hat{y} = Xw + b$$
Minimizing Mean Squared Error (MSE):
$$J(w, b) = \frac{1}{N} \sum_{i=1}^N (y_i - \hat{y}_i)^2$$

### Why Linear Regression Fails for Classification:
1. **Unbounded Output Space**: $\hat{y} \in (-\infty, +\infty)$. It outputs values $< 0$ and $> 1$, which violate probability axioms.
2. **MSE "Over-Penalty" on Outliers**: If a message is overwhelmingly spammy (e.g., $w^T x = 3.0$), true label is $y=1$. Linear regression incurs $(1 - 3.0)^2 = 4.0$ squared error! It pulls the decision boundary toward the spam class, causing **False Positives** on legitimate messages.


In [ ]:
lin_reg = LinearRegression()
lin_reg.fit(X_train_tfidf, y_train)

y_pred_cont = lin_reg.predict(X_test_tfidf)

y_pred_lin = (y_pred_cont >= 0.5).astype(int)

min_pred = float(np.min(y_pred_cont))
max_pred = float(np.max(y_pred_cont))
neg_count = int(np.sum(y_pred_cont < 0))
over_count = int(np.sum(y_pred_cont > 1))
total = len(y_pred_cont)

print(f"Linear Regression Continuous Output Analysis (N = {total}):")
print(f"  - Minimum prediction: {min_pred:+.4f} (Violates P >= 0!)")
print(f"  - Maximum prediction: {max_pred:+.4f} (Violates P <= 1!)")
print(f"  - Samples with P < 0: {neg_count} ({neg_count/total*100:.2f}%)")
print(f"  - Samples with P > 1: {over_count} ({over_count/total*100:.2f}%)")
print(f"  - Total Probability Violations: {neg_count + over_count} ({(neg_count+over_count)/total*100:.2f}%)")


## 5. Model 2: Logistic Regression (Proper Linear Classifier)

### Mathematical Formulation:
1. **Log-Odds / Logit**:
   $$z = \ln\left(\frac{p}{1-p}\right) = w^T x + b$$
2. **Sigmoid Activation Function**:
   $$p = \sigma(z) = \frac{1}{1 + e^{-z}} \in (0, 1)$$
   - Maps $(-\infty, +\infty)$ smoothly to $(0, 1)$.
   - $\sigma(0) = 0.5$ defines the decision boundary hyperplane $w^T x + b = 0$.
3. **Log-Loss (Binary Cross-Entropy)**:
   $$J(w) = -\frac{1}{N} \sum_{i=1}^N \left[ y_i \ln(p_i) + (1 - y_i) \ln(1 - p_i) \right]$$
   - Convex optimization landscape.
   - High confidence in the correct class yields near-zero loss without penalizing outliers.


In [ ]:
log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
log_reg.fit(X_train_tfidf, y_train)

y_pred_log = log_reg.predict(X_test_tfidf)
y_proba_log = log_reg.predict_proba(X_test_tfidf)[:, 1]
print("Logistic Regression fitted successfully.")


## 6. Model 3: Multinomial Naive Bayes (Standard Text Benchmark)

### Mathematical Formulation:
1. **Bayes' Theorem**:
   $$P(c | d) = \frac{P(d | c) P(c)}{P(d)} \propto P(c) \prod_{i=1}^V P(w_i | c)^{f_i}$$
2. **Conditional Independence Assumption ("Naive")**:
   Assumes words appear independently of one another given the document class.
3. **Laplace (Additive) Smoothing ($\alpha = 1.0$)**:
   $$\hat{P}(w_i | c) = \frac{N_{c, i} + \alpha}{N_c + \alpha |V|}$$
   Prevents zero-probability collapse when test documents contain tokens unseen in a training class.


In [ ]:
nb_model = MultinomialNB(alpha=1.0)
nb_model.fit(X_train_tfidf, y_train)

y_pred_nb = nb_model.predict(X_test_tfidf)
y_proba_nb = nb_model.predict_proba(X_test_tfidf)[:, 1]
print("Multinomial Naive Bayes fitted successfully.")


## 7. Model Evaluation & Comparison Table

We compute standard metrics on the held-out test set ($N=1115$):
- **Accuracy**: $\frac{TP + TN}{\text{Total}}$
- **Precision**: $\frac{TP}{TP + FP}$
- **Recall**: $\frac{TP}{TP + FN}$
- **F1-Score**: $2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$
- **Confusion Matrix**: $[[TN, FP], [FN, TP]]$


In [ ]:
models = [
    ('1. Linear Regression (Threshold 0.5)', y_pred_lin),
    ('2. Logistic Regression (Sigmoid)', y_pred_log),
    ('3. Multinomial Naive Bayes (Baseline)', y_pred_nb)
]

results = []
for name, preds in models:
    cm = confusion_matrix(y_test, preds)
    tn, fp, fn, tp = cm.ravel()
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds),
        'Recall': recall_score(y_test, preds),
        'F1-Score': f1_score(y_test, preds),
        'False Positives (FP)': fp,
        'False Negatives (FN)': fn
    })

results_df = pd.DataFrame(results)
display_df = results_df.copy()
display_df['Accuracy'] = display_df['Accuracy'].map('{:.2%}'.format)
display_df['Precision'] = display_df['Precision'].map('{:.2%}'.format)
display_df['Recall'] = display_df['Recall'].map('{:.2%}'.format)
display_df['F1-Score'] = display_df['F1-Score'].map('{:.2%}'.format)

print("\n--- MODEL PERFORMANCE COMPARISON TABLE ---")
print(display_df.to_string(index=False))


## 8. Why Precision is the Most Critical Metric for Spam Filtering

### Asymmetric Costs of Errors:
| Error Type | Description | Consequence | Severity |
| :--- | :--- | :--- | :--- |
| **False Positive (Type I)** | Legitimate email (Ham) classified as **Spam** | Job offer, password reset OTP, medical alert, or client contract is routed to Spam folder and missed. | **CATASTROPHIC** |
| **False Negative (Type II)** | Spam email classified as **Ham** | A promo or scam email lands in the inbox. User clicks delete in 1 second. | **MINOR INCONVENIENCE** |

$$\text{Precision} = \frac{TP}{TP + FP}$$

Maximizing precision directly drives **False Positives towards 0**.
Notice how **Logistic Regression** achieved **99.19% Precision** with only **1 False Positive**, whereas Linear Regression produced **12 False Positives**!


## 9. Sample Prediction Function: `predict_message(text)`

Interactive CLI helper function that preprocesses custom text, vectorizes it, and returns the classification and calibrated probability.


In [ ]:
def predict_message(text: str, model=log_reg, threshold: float = 0.5) -> dict:
    cleaned = clean_text(text)
    vec = vectorizer.transform([cleaned])
    
    if hasattr(model, "predict_proba"):
        prob = float(model.predict_proba(vec)[0, 1])
        label = 1 if prob >= threshold else 0
    else:
        score = float(model.predict(vec)[0])
        prob = score
        label = 1 if score >= threshold else 0
        
    return {
        "text": text,
        "prediction": "SPAM" if label == 1 else "HAM",
        "spam_probability": prob
    }

sample_messages = [
    "Hey, can we sync on the project deadline tomorrow afternoon?",
    "URGENT! Your bank account has been compromised. Log in at http://fake-bank.com to protect your funds.",
    "Your Uber verification code is 591024. Valid for 10 minutes.",
    "WINNER! You have won a free $1,000 Amazon Gift Card! Text WIN to 80808 now!"
]

for msg in sample_messages:
    res = predict_message(msg)
    print(f"\nMessage:    '{res['text']}'")
    print(f"Prediction: {res['prediction']} (P(Spam) = {res['spam_probability']:.2%})")
